# Dual-scale persistent-map CNN-DQN for MiniGrid exploration

This notebook validates and trains the pipeline:

```text
partial observation → persistent mapper → dual-scale tensors
→ local/global CNN branches → fused DQN → 7 Q-values
```

Only `left`, `right`, and `forward` are selectable during this exploration stage. The sole learning reward is `exploration_beta * MappingUpdate.new_cells`.

## Imports and configuration

In [ ]:
from collections import Counter
from copy import deepcopy
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim

from minigrid.core.constants import OBJECT_TO_IDX

from utils import alert_training_complete
from utils.dqn_utils import hard_update_target, linear_epsilon, set_seed
from utils.minigrid_dual_scale_dqn import (
    DualScaleMapDQN,
    DualScaleReplayBuffer,
    EXPLORATION_ACTIONS,
    dual_scale_dqn_train_step,
    load_dual_scale_checkpoint,
    make_dual_scale_state,
    mask_q_values,
    save_dual_scale_checkpoint,
    select_masked_epsilon_greedy_action,
    select_masked_greedy_action,
)
from utils.minigrid_map_encoder import PersistentMapTensorEncoder
from utils.minigrid_mapper import MapCell, PersistentMiniGridMapper
from utils.procedural_rooms_env import ENVIRONMENT_ID

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Tensor and network configuration
local_size = 15
global_size = 15
unknown_padding = 2
hysteresis_bins = 1
conv_channels = (32, 64, 64)
fusion_hidden_size = 256

# Procedural training distribution
training_environment_config = {
    "width": 19,
    "height": 19,
    "min_rooms": 4,
    "max_rooms": 6,
    "min_room_size": 5,
    "doorway_width": 1,
    "max_steps": 400,
}

# Disjoint procedural seed ranges
training_seed_start = 0
training_seed_count = 10_000
validation_seeds = list(range(20_000, 20_100))
test_seeds = list(range(30_000, 30_200))
generalization_seeds = list(range(40_000, 40_100))

# Long training is intentionally opt-in after visual validation.
RUN_TRAINING = False
RUN_EVALUATION = False

## Visual validation of local and global tensors

The mapper is allowed to accumulate knowledge through random exploration actions. Hidden map state is not used. The displayed tensors are exactly what the CNN receives.

In [ ]:
visual_env = gym.make(ENVIRONMENT_ID, max_steps=120)
observation, info = visual_env.reset(seed=1234)
visual_env.action_space.seed(1234)
visual_mapper = PersistentMiniGridMapper()
visual_mapper.reset(observation)
visual_encoder = PersistentMapTensorEncoder(
    local_size=local_size,
    global_size=global_size,
    unknown_padding=unknown_padding,
    hysteresis_bins=hysteresis_bins,
)
visual_encoder.reset()

for _ in range(100):
    action = visual_env.action_space.sample() % 3
    visual_mapper.predict_action(action)
    observation, reward, terminated, truncated, info = visual_env.step(
        action
    )
    visual_mapper.observe(observation)

    if terminated or truncated:
        break

encoded_visual_map = visual_encoder.encode(visual_mapper)
visual_env.close()

print("Global scale:", encoded_visual_map.global_scale)
print("Known map bounds:", visual_mapper.known_bounds(padding=0))
print("Agent position:", tuple(visual_mapper.position))
print("Agent direction:", visual_mapper.direction)
print("Local tensor shape:", encoded_visual_map.local.shape)
print("Global tensor shape:", encoded_visual_map.global_map.shape)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for index, channel_name in enumerate(visual_encoder.LOCAL_CHANNELS):
    axes[0, index].imshow(encoded_visual_map.local[index], vmin=0, vmax=1)
    axes[0, index].set_title(f"Local: {channel_name}")
    axes[0, index].axis("off")

axes[0, 4].axis("off")

for index, channel_name in enumerate(visual_encoder.GLOBAL_CHANNELS):
    axes[1, index].imshow(
        encoded_visual_map.global_map[index], vmin=0, vmax=1
    )
    axes[1, index].set_title(f"Global: {channel_name}")
    axes[1, index].axis("off")

plt.tight_layout()
plt.show()

## Orientation normalization

The same persistent world map is encoded under all four compass directions. The geometry rotates so tensor-up is always forward, while the agent remains fixed at the center.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
direction_names = ("east", "south", "west", "north")

for direction, axis in enumerate(axes):
    rotated_mapper = deepcopy(visual_mapper)
    rotated_mapper.direction = direction
    rotated_encoder = PersistentMapTensorEncoder(
        local_size=local_size, global_size=global_size
    )
    rotated = rotated_encoder.encode(rotated_mapper)
    axis.imshow(rotated.local[1], vmin=0, vmax=1)
    axis.scatter([local_size // 2], [local_size // 2], c="red", s=25)
    axis.set_title(f"Facing {direction_names[direction]}")
    axis.axis("off")

plt.suptitle("Local wall channel; tensor-up is forward")
plt.tight_layout()
plt.show()

## Targeted adaptive-scale and hysteresis tests

In [ ]:
class SyntheticMapper:
    """Minimal mapper interface for deterministic encoder tests."""

    def __init__(self):
        self.position = np.array([0, 0], dtype=int)
        self.direction = 0
        self.cells = {}

    def frontier_cells(self):
        return set()


empty_cell = MapCell(
    object_idx=OBJECT_TO_IDX["empty"], color_idx=0, state=0
)
synthetic_mapper = SyntheticMapper()
scale_encoder = PersistentMapTensorEncoder(
    global_size=15, unknown_padding=2, hysteresis_bins=1
)

scale_cases = [
    ("comfortably fits", 4, 1),
    ("approaches boundary", 5, 1),
    ("padding barely exceeds nominal radius", 6, 1),
    ("requires scale 2", 7, 2),
    ("requires scale 4", 15, 4),
]
observed_scales = []

for label, extent, expected_scale in scale_cases:
    synthetic_mapper.cells = {
        (x, 0): empty_cell for x in range(-extent, extent + 1)
    }
    encoded = scale_encoder.encode(synthetic_mapper)
    observed_scales.append(encoded.global_scale)
    assert encoded.global_scale == expected_scale, label
    print(f"{label:40s} extent={extent:2d} scale={encoded.global_scale}")

# Shrinking the known map cannot reduce scale within the episode.
synthetic_mapper.cells = {(0, 0): empty_cell}
assert scale_encoder.encode(synthetic_mapper).global_scale == 4

# A new episode explicitly resets scale to exact resolution.
scale_encoder.reset()
small_encoding = scale_encoder.encode(synthetic_mapper)
assert small_encoding.global_scale == 1
assert np.all((small_encoding.global_map[:4] == 0) | (small_encoding.global_map[:4] == 1))

print("Scale sequence:", observed_scales)
print("No oscillation and scale-1 binary-density checks passed.")

## CNN forward, masking, and gradient tests

In [ ]:
test_network = DualScaleMapDQN(
    action_dim=7,
    conv_channels=conv_channels,
    fusion_hidden_size=fusion_hidden_size,
).to(device)

for test_batch_size in (1, 32, 64):
    local_batch = torch.rand(test_batch_size, 4, 15, 15, device=device)
    global_batch = torch.rand(test_batch_size, 5, 15, 15, device=device)
    scale_batch = torch.randint(
        0, 4, (test_batch_size, 1), device=device
    ).float()
    q_values = test_network(local_batch, global_batch, scale_batch)

    assert q_values.shape == (test_batch_size, 7)
    assert torch.all(torch.isfinite(q_values))
    assert torch.all(
        torch.isneginf(mask_q_values(q_values)[:, 3:])
    )

test_network.zero_grad()
q_values.sum().backward()
gradient_groups = {
    "local CNN": list(test_network.local_cnn.parameters()),
    "global CNN": list(test_network.global_cnn.parameters()),
    "fusion": list(test_network.fusion.parameters()),
    "DQN head": list(test_network.dqn_head.parameters()),
}

for group_name, parameters in gradient_groups.items():
    assert all(parameter.grad is not None for parameter in parameters)
    assert all(torch.all(torch.isfinite(parameter.grad)) for parameter in parameters)
    print(group_name, "gradients: OK")

print("Batch sizes 1, 32, and 64 produced finite (B, 7) outputs.")

## Evaluation helpers

These helpers use the same seeds, step budget, mapper, encoder, coverage calculation, and allowed actions for learned and random controllers. They do not hide the training loop.

In [ ]:
coverage_thresholds = (0.50, 0.75, 0.90)


def known_traversable_cells(mapper):
    return sum(
        cell.object_name in {"empty", "floor"}
        for cell in mapper.cells.values()
    )


def finite_mean(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    return float(np.mean(finite)) if finite.size else np.nan


def summarize_evaluations(results):
    summary = {
        "mean_final_coverage": np.mean(
            [result["final_coverage"] for result in results]
        ),
        "median_final_coverage": np.median(
            [result["final_coverage"] for result in results]
        ),
        "mean_traversable_coverage": np.mean(
            [result["traversable_coverage"] for result in results]
        ),
        "new_cells_per_step": np.mean(
            [result["new_cells_per_step"] for result in results]
        ),
        "zero_information_ratio": np.mean(
            [result["zero_information_ratio"] for result in results]
        ),
    }

    for threshold in coverage_thresholds:
        summary[f"steps_to_{int(100 * threshold)}"] = finite_mean(
            [result["steps_to_coverage"][threshold] for result in results]
        )

    return summary


def evaluate_exploration_controller(
    seeds,
    environment_config,
    policy_net=None,
    random_controller=False,
):
    results = []

    if policy_net is not None:
        policy_net.eval()

    for seed in seeds:
        evaluation_env = gym.make(ENVIRONMENT_ID, **environment_config)
        observation, info = evaluation_env.reset(seed=seed)
        mapper = PersistentMiniGridMapper()
        mapper.reset(observation)
        encoder = PersistentMapTensorEncoder(
            local_size=local_size,
            global_size=global_size,
            unknown_padding=unknown_padding,
            hysteresis_bins=hysteresis_bins,
        )
        encoder.reset()
        state = make_dual_scale_state(encoder.encode(mapper))
        action_rng = np.random.default_rng(900_000 + seed)
        total_map_cells = evaluation_env.unwrapped.width * evaluation_env.unwrapped.height
        total_traversable = evaluation_env.unwrapped.total_traversable_cells
        coverage_history = [len(mapper.cells) / total_map_cells]
        steps_to_coverage = {threshold: np.nan for threshold in coverage_thresholds}
        new_cell_counts = []
        terminated = False
        truncated = False
        steps = 0

        while not (terminated or truncated):
            if random_controller:
                action = int(action_rng.choice(EXPLORATION_ACTIONS))
            else:
                action = select_masked_greedy_action(
                    policy_net, state, device
                )

            mapper.predict_action(action)
            observation, env_reward, terminated, truncated, info = (
                evaluation_env.step(action)
            )
            update = mapper.observe(observation)
            state = make_dual_scale_state(encoder.encode(mapper))
            steps += 1
            new_cell_counts.append(update.new_cells)
            coverage = len(mapper.cells) / total_map_cells
            coverage_history.append(coverage)

            for threshold in coverage_thresholds:
                if np.isnan(steps_to_coverage[threshold]) and coverage >= threshold:
                    steps_to_coverage[threshold] = float(steps)

        new_cell_counts = np.asarray(new_cell_counts, dtype=float)
        results.append(
            {
                "seed": seed,
                "coverage_history": np.asarray(coverage_history),
                "final_coverage": coverage_history[-1],
                "traversable_coverage": (
                    known_traversable_cells(mapper) / total_traversable
                ),
                "steps_to_coverage": steps_to_coverage,
                "new_cells_per_step": np.mean(new_cell_counts),
                "zero_information_ratio": np.mean(new_cell_counts == 0),
            }
        )
        evaluation_env.close()

    return results

## Intrinsic-exploration DQN training

The full episode loop remains visible. Initial mapping establishes knowledge without reward. Every replay state is an immutable copy of local/global tensors and `log2(global_scale)`. Set `RUN_TRAINING = True` only after the representations above look correct.

In [ ]:
num_training_episodes = 2_000
exploration_beta = 1.0
gamma = 0.99
learning_rate = 1e-4
batch_size = 64
replay_capacity = 100_000
minimum_replay_size = 5_000
target_update_frequency = 1_000
gradient_clip = 10.0
epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay_steps = 300_000
validation_frequency = 100
validation_subset = validation_seeds[:20]
training_random_seed = 42
model_path = Path("models") / "minigrid_dual_scale_explorer.pth"

In [ ]:
if RUN_TRAINING:
    set_seed(training_random_seed)
    training_rng = np.random.default_rng(training_random_seed)
    training_env = gym.make(ENVIRONMENT_ID, **training_environment_config)
    policy_net = DualScaleMapDQN(
        action_dim=7,
        conv_channels=conv_channels,
        fusion_hidden_size=fusion_hidden_size,
    ).to(device)
    target_net = DualScaleMapDQN(
        action_dim=7,
        conv_channels=conv_channels,
        fusion_hidden_size=fusion_hidden_size,
    ).to(device)
    hard_update_target(policy_net, target_net)
    target_net.eval()
    optimizer = optim.Adam(policy_net.parameters(), lr=learning_rate)
    replay_buffer = DualScaleReplayBuffer(replay_capacity)
    training_metrics = []
    validation_history = []
    total_steps = 0

    for episode in range(num_training_episodes):
        episode_seed = training_seed_start + (episode % training_seed_count)
        observation, info = training_env.reset(seed=episode_seed)
        mapper = PersistentMiniGridMapper()
        mapper.reset(observation)
        encoder = PersistentMapTensorEncoder(
            local_size=local_size,
            global_size=global_size,
            unknown_padding=unknown_padding,
            hysteresis_bins=hysteresis_bins,
        )
        encoder.reset()
        state = make_dual_scale_state(encoder.encode(mapper))

        total_map_cells = training_env.unwrapped.width * training_env.unwrapped.height
        total_traversable = training_env.unwrapped.total_traversable_cells
        episode_intrinsic_reward = 0.0
        episode_losses = []
        new_cell_counts = []
        scale_counts = Counter([int(2 ** state.global_scale_log2)])
        steps_to_coverage = {threshold: np.nan for threshold in coverage_thresholds}
        episode_steps = 0
        terminated = False
        truncated = False

        while not (terminated or truncated):
            epsilon = linear_epsilon(
                total_steps, epsilon_start, epsilon_end, epsilon_decay_steps
            )
            action = select_masked_epsilon_greedy_action(
                policy_net,
                state,
                epsilon,
                device,
                random_generator=training_rng,
            )

            mapper.predict_action(action)
            observation, env_reward, terminated, truncated, info = (
                training_env.step(action)
            )
            update = mapper.observe(observation)
            intrinsic_reward = exploration_beta * update.new_cells
            next_state = make_dual_scale_state(encoder.encode(mapper))
            done = terminated or truncated
            replay_buffer.push(
                state, action, intrinsic_reward, next_state, done
            )

            loss = dual_scale_dqn_train_step(
                policy_net,
                target_net,
                optimizer,
                replay_buffer,
                batch_size,
                gamma,
                device,
                min_replay_size=minimum_replay_size,
                gradient_clip=gradient_clip,
            )

            if loss is not None:
                episode_losses.append(loss)

            total_steps += 1
            episode_steps += 1
            episode_intrinsic_reward += intrinsic_reward
            new_cell_counts.append(update.new_cells)
            scale_counts[int(2 ** next_state.global_scale_log2)] += 1
            state = next_state

            if total_steps % target_update_frequency == 0:
                hard_update_target(policy_net, target_net)

            map_coverage = len(mapper.cells) / total_map_cells

            for threshold in coverage_thresholds:
                if np.isnan(steps_to_coverage[threshold]) and map_coverage >= threshold:
                    steps_to_coverage[threshold] = float(episode_steps)

        new_cell_counts = np.asarray(new_cell_counts, dtype=float)
        local_tensor = torch.tensor(
            state.local, dtype=torch.float32, device=device
        ).unsqueeze(0)
        global_tensor = torch.tensor(
            state.global_map, dtype=torch.float32, device=device
        ).unsqueeze(0)
        scale_tensor = torch.tensor(
            [[state.global_scale_log2]], dtype=torch.float32, device=device
        )

        with torch.no_grad():
            predicted_q = mask_q_values(
                policy_net(local_tensor, global_tensor, scale_tensor)
            )[:, list(EXPLORATION_ACTIONS)]

        training_metrics.append(
            {
                "episode": episode + 1,
                "intrinsic_reward": episode_intrinsic_reward,
                "final_coverage": len(mapper.cells) / total_map_cells,
                "traversable_coverage": (
                    known_traversable_cells(mapper) / total_traversable
                ),
                "new_cells_per_step": np.mean(new_cell_counts),
                "zero_information_ratio": np.mean(new_cell_counts == 0),
                "steps_to_coverage": steps_to_coverage,
                "episode_length": episode_steps,
                "epsilon": epsilon,
                "training_loss": (
                    np.mean(episode_losses) if episode_losses else np.nan
                ),
                "mean_predicted_q": float(predicted_q.mean().item()),
                "max_predicted_q": float(predicted_q.max().item()),
                "global_scale_distribution": dict(scale_counts),
            }
        )

        if (episode + 1) % 25 == 0:
            recent = training_metrics[-25:]
            print(
                f"Episode {episode + 1:4d}/{num_training_episodes} | "
                f"coverage={np.mean([m['final_coverage'] for m in recent]):.1%} | "
                f"reward={np.mean([m['intrinsic_reward'] for m in recent]):.1f} | "
                f"epsilon={epsilon:.3f} | loss={finite_mean([m['training_loss'] for m in recent]):.4f}"
            )

        if (episode + 1) % validation_frequency == 0:
            validation_results = evaluate_exploration_controller(
                validation_subset, training_environment_config, policy_net
            )
            validation_summary = summarize_evaluations(validation_results)
            validation_history.append(
                {"episode": episode + 1, **validation_summary}
            )
            policy_net.train()
            print(
                f"Validation coverage: "
                f"{validation_summary['mean_final_coverage']:.1%}"
            )

    training_env.close()
    model_path.parent.mkdir(parents=True, exist_ok=True)
    save_dual_scale_checkpoint(
        model_path,
        policy_net,
        target_net=target_net,
        optimizer=optimizer,
        metadata={
            "training_episode": len(training_metrics),
            "total_steps": total_steps,
            "epsilon": epsilon,
            "tensor_config": {
                "local_size": local_size,
                "global_size": global_size,
                "unknown_padding": unknown_padding,
                "hysteresis_bins": hysteresis_bins,
            },
            "exploration_beta": exploration_beta,
            "training_environment_config": training_environment_config,
            "training_seed_range": (
                training_seed_start,
                training_seed_start + training_seed_count - 1,
            ),
            "validation_seed_range": (
                validation_seeds[0], validation_seeds[-1]
            ),
            "test_seed_range": (test_seeds[0], test_seeds[-1]),
            "hyperparameters": {
                "gamma": gamma,
                "learning_rate": learning_rate,
                "batch_size": batch_size,
                "epsilon_start": epsilon_start,
                "epsilon_end": epsilon_end,
                "epsilon_decay_steps": epsilon_decay_steps,
                "allowed_actions": EXPLORATION_ACTIONS,
            },
            "training_metrics": training_metrics,
            "validation_history": validation_history,
        },
    )
    alert_training_complete(
        "Dual-scale MiniGrid exploration training finished."
    )
    print("Saved checkpoint to:", model_path)
else:
    print("Tensor/CNN validation complete. Set RUN_TRAINING = True to train.")

## Training diagnostics

In [ ]:
if RUN_TRAINING:
    episode_axis = np.arange(1, len(training_metrics) + 1)
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    axes[0, 0].plot(
        episode_axis, [m["intrinsic_reward"] for m in training_metrics]
    )
    axes[0, 0].set_title("Intrinsic exploration reward")

    axes[0, 1].plot(
        episode_axis, [m["final_coverage"] for m in training_metrics],
        label="Map",
    )
    axes[0, 1].plot(
        episode_axis,
        [m["traversable_coverage"] for m in training_metrics],
        label="Traversable",
    )
    axes[0, 1].set_title("Final coverage")
    axes[0, 1].legend()

    axes[0, 2].plot(
        episode_axis, [m["new_cells_per_step"] for m in training_metrics]
    )
    axes[0, 2].set_title("New cells per step")

    axes[0, 3].plot(
        episode_axis,
        [m["zero_information_ratio"] for m in training_metrics],
    )
    axes[0, 3].set_title("Zero-information step ratio")

    axes[1, 0].plot(
        episode_axis, [m["training_loss"] for m in training_metrics]
    )
    axes[1, 0].set_title("Training loss")

    axes[1, 1].plot(
        episode_axis, [m["mean_predicted_q"] for m in training_metrics],
        label="Mean",
    )
    axes[1, 1].plot(
        episode_axis, [m["max_predicted_q"] for m in training_metrics],
        label="Max",
    )
    axes[1, 1].set_title("Allowed-action Q-values")
    axes[1, 1].legend()

    axes[1, 2].plot(
        episode_axis, [m["epsilon"] for m in training_metrics]
    )
    axes[1, 2].set_title("Epsilon")

    observed_scales = sorted(
        {
            scale
            for metric in training_metrics
            for scale in metric["global_scale_distribution"]
        }
    )
    scale_fractions = []

    for scale in observed_scales:
        scale_fractions.append(
            [
                metric["global_scale_distribution"].get(scale, 0)
                / sum(metric["global_scale_distribution"].values())
                for metric in training_metrics
            ]
        )

    axes[1, 3].stackplot(
        episode_axis, *scale_fractions, labels=[f"s={s}" for s in observed_scales]
    )
    axes[1, 3].set_title("Global-scale distribution")
    axes[1, 3].legend()

    for axis in axes.flat:
        axis.set_xlabel("Episode")
        axis.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

## Unseen-map evaluation and random baseline

Set `RUN_EVALUATION = True` after a checkpoint exists. Both controllers run on the same held-out seeds with identical map distributions and 400-step budgets.

In [ ]:
if RUN_EVALUATION:
    if not model_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {model_path.resolve()}")

    evaluation_policy, checkpoint = load_dual_scale_checkpoint(
        model_path, device
    )
    learned_results = evaluate_exploration_controller(
        test_seeds, training_environment_config, evaluation_policy
    )
    random_results = evaluate_exploration_controller(
        test_seeds,
        training_environment_config,
        random_controller=True,
    )
    learned_summary = summarize_evaluations(learned_results)
    random_summary = summarize_evaluations(random_results)

    print("Held-out 19x19 test results")
    print("Metric                         Learned       Random")
    for metric in learned_summary:
        print(
            f"{metric:30s} "
            f"{learned_summary[metric]:10.4f}   "
            f"{random_summary[metric]:10.4f}"
        )

    learned_curve = np.mean(
        np.stack([result["coverage_history"] for result in learned_results]),
        axis=0,
    )
    random_curve = np.mean(
        np.stack([result["coverage_history"] for result in random_results]),
        axis=0,
    )
    plt.figure(figsize=(10, 5))
    plt.plot(100 * learned_curve, label="Dual-scale CNN-DQN")
    plt.plot(100 * random_curve, label="Random baseline")
    plt.xlabel("Environment step")
    plt.ylabel("Mean map coverage (%)")
    plt.title("Coverage on identical unseen procedural maps")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Set RUN_EVALUATION = True after training to compare controllers.")

## Zero-shot 27×27 generalization

The same trained network and tensor sizes are evaluated without retraining. Only the procedural environment dimensions and step budget change.

In [ ]:
if RUN_EVALUATION:
    larger_environment_config = {
        **training_environment_config,
        "width": 27,
        "height": 27,
        "max_steps": 800,
    }
    larger_learned_results = evaluate_exploration_controller(
        generalization_seeds, larger_environment_config, evaluation_policy
    )
    larger_random_results = evaluate_exploration_controller(
        generalization_seeds,
        larger_environment_config,
        random_controller=True,
    )
    print("27x27 learned:", summarize_evaluations(larger_learned_results))
    print("27x27 random:", summarize_evaluations(larger_random_results))